# Data Utilities
Load the Wisconsin Diagnostic Breast Cancer dataset, discretize numeric features, and apply random feature masking to simulate partial observability.

In [22]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath(".."), "src"))

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

## Load Breast Cancer Dataset (Wisconsin Diagnostic)

In [23]:
def load_breast_cancer_data(n_bins=10):
    """
    Load the Wisconsin Diagnostic Breast Cancer dataset (sklearn built-in).
    Numeric features are discretized into `n_bins` quantile bins so that
    the categorical Naive Bayes and SEU framework can be applied directly.

    Returns (X, y) as numpy int arrays.
      X : (569, 30) discretized feature matrix
      y : (569,)    0 = malignant, 1 = benign
    """
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer()
    disc = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile",
                               quantile_method="averaged_inverted_cdf")
    X = disc.fit_transform(data.data).astype(int)
    y = data.target
    return X, y

## Apply Random Mask

In [24]:
def apply_mask(X, missing_rate, rng):
    """
    Randomly mask `missing_rate` fraction of feature entries with NaN.

    Parameters
    ----------
    X : pd.DataFrame  (integer-encoded, no NaNs)
    missing_rate : float  e.g. 0.10 or 0.20
    rng : np.random.Generator

    Returns
    -------
    X_masked : pd.DataFrame  (float dtype; NaN where masked)
    mask : np.ndarray bool   True where value is MISSING
    """
    X_masked = X.astype(float).copy()
    n_rows, n_cols = X_masked.shape
    total_entries = n_rows * n_cols
    n_missing = int(total_entries * missing_rate)

    flat_indices = rng.choice(total_entries, size=n_missing, replace=False)
    row_idx, col_idx = np.unravel_index(flat_indices, (n_rows, n_cols))

    mask = np.zeros((n_rows, n_cols), dtype=bool)
    for r, c in zip(row_idx, col_idx):
        X_masked.iat[r, c] = np.nan
        mask[r, c] = True

    return X_masked, mask

In [25]:
X, y = load_breast_cancer_data()
print(f"Breast Cancer (Wisconsin): {X.shape[0]} instances, {X.shape[1]} features")
print(f"  Class distribution: {np.bincount(y)}  (0=malignant, 1=benign)")

Breast Cancer (Wisconsin): 569 instances, 30 features
  Class distribution: [212 357]  (0=malignant, 1=benign)


In [26]:
rng = np.random.default_rng(42)
X_m10, mask10 = apply_mask(pd.DataFrame(X), 0.10, rng)
rng2 = np.random.default_rng(42)
X_m20, mask20 = apply_mask(pd.DataFrame(X), 0.20, rng2)
print(f"10% mask: {mask10.sum()} entries ({mask10.mean()*100:.1f}%)")
print(f"20% mask: {mask20.sum()} entries ({mask20.mean()*100:.1f}%)")

10% mask: 1707 entries (10.0%)
20% mask: 3414 entries (20.0%)
